# Lab 3: PCA si K-Means Clustering (Ames Housing)

## 1. Configurarea Experimentelor

Analizam datasetul **Ames Housing** folosind *Principal Component Analysis (PCA)* pentru reducerea dimensionalitatii si **K-means** pentru identificarea grupurilor de case cu proprietati similare.

* **Praguri PCA:** pastram **98%, 95%, 90% si 80%** din varianta datelor procesate.
* **Numarul de clustere:** testam valori ale lui `k` intre **2 si 10**.
* **Reproductibilitate:** folosim `SEED = 42` pentru initializarea algoritmilor.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import MinMaxScaler, Normalizer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score

SEED = 42
DATA_PATH = Path('../data/AmesHousing.csv')
THRESHOLDS = [0.98, 0.95, 0.90, 0.80]
K_VALUES = list(range(2, 11))
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)


## 2. Pregatirea Datelor si Strategii de Preprocesare

Pornim de la regulile de curatare folosite in L1 si L2. Pastram atributele originale pentru a putea interpreta ulterior caracteristicile fiecarui cluster.

1. **Eliminarea outlierilor:** excludem casele cu `Gr Liv Area > 4000` si `SalePrice < 300000`, conform regulii din laboratoarele anterioare.
2. **Separarea atributelor:** eliminam `SalePrice`, `Order` si `PID` din intrarea modelului. Pretul ramane disponibil pentru descrierea clusterelor dupa antrenare.
3. **Tratarea valorilor lipsa:** folosim `None` cand lipsa inseamna absenta unei dotari, mediana pentru atributele numerice si categoria cea mai frecventa pentru celelalte valori categorice lipsa.
4. **Encoding:** transformarile ordinale pastreaza ordinea calificarilor, iar *One-Hot Encoding* creeaza coloane separate pentru categoriile nominale. `MS SubClass` este tratat ca o categorie.
5. **Selectia atributelor pentru logaritmare:** alegem masuratori nenegative cu asimetrie mai mare decat 1. Aplicam ulterior `log1p`, care accepta si valoarea zero.

Definim functii reutilizabile de curatare si preprocesare. Atributele combinate din L2 nu sunt adaugate aici, deoarece repetarea informatiei despre suprafata sau calitate poate modifica distantele folosite de K-means.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Place AmesHousing.csv at {DATA_PATH.resolve()} or edit DATA_PATH.')
raw = pd.read_csv(DATA_PATH, sep=None, engine='python')
raw.columns = raw.columns.str.strip()
required = {'SalePrice', 'Gr Liv Area', 'Overall Qual', 'Year Built'}
assert required.issubset(raw.columns), 'Expected original Ames column names with spaces.'
outliers = (raw['Gr Liv Area'] > 4000) & (raw['SalePrice'] < 300000)
df = raw.loc[~outliers].reset_index(drop=True).copy()
X_raw = df.drop(columns=['SalePrice', 'Order', 'PID'], errors='ignore')
print(f'{len(df)} houses; removed {outliers.sum()} outliers; {X_raw.shape[1]} raw features.')

ABSENCE = ['Alley', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1',
           'BsmtFin Type 2', 'Fireplace Qu', 'Garage Type', 'Garage Finish',
           'Garage Qual', 'Garage Cond', 'Pool QC', 'Fence', 'Misc Feature', 'Mas Vnr Type']
QUALITY = ['Exter Qual', 'Exter Cond', 'Bsmt Qual', 'Bsmt Cond', 'Heating QC',
           'Kitchen Qual', 'Fireplace Qu', 'Garage Qual', 'Garage Cond', 'Pool QC']
def clean_features(frame):
    x = frame.copy()
    for col in ABSENCE:
        if col in x: x[col] = x[col].fillna('None')
    for col in QUALITY:
        if col in x:
            x[col] = x[col].map({'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5})
    if 'MS SubClass' in x: x['MS SubClass'] = x['MS SubClass'].astype(str)
    return x

X = clean_features(X_raw)
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = [col for col in X if col not in num_cols]
# Log only nonnegative, strongly right-skewed numerical measurements; exclude ratings and years.
log_candidates = ['Lot Frontage', 'Lot Area', 'Mas Vnr Area', 'BsmtFin SF 1',
                  'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF', '1st Flr SF',
                  '2nd Flr SF', 'Gr Liv Area', 'Garage Area', 'Wood Deck SF',
                  'Open Porch SF', 'Enclosed Porch', '3Ssn Porch', 'Screen Porch',
                  'Pool Area', 'Misc Val']
LOG_COLS = [col for col in log_candidates if col in X and X[col].min() >= 0 and X[col].skew() > 1]
print('Log1p features:', LOG_COLS)

def make_preprocessor():
    return ColumnTransformer([
        ('numeric', SimpleImputer(strategy='median'), num_cols),
        ('categorical', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent')),
            ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols)
    ], verbose_feature_names_out=False)

def before_encoding(frame, use_log):
    x = clean_features(frame)
    if use_log: x[LOG_COLS] = np.log1p(x[LOG_COLS])
    return x


## 3. Reducerea Dimensionalitatii cu PCA si Compararea Scalarii

Comparam **patru strategii de preprocesare**, fiecare evaluata la cele patru praguri PCA:

* **Min-Max Scaling:** rescalam separat fiecare atribut in intervalul `[0, 1]` pentru datele folosite la antrenare.
* **Normalizare L2:** impartim fiecare rand la norma sa, astfel incat vectorul fiecarei case sa aiba lungimea 1.
* **Log1p + Min-Max:** reducem influenta valorilor foarte mari din atributele selectate, apoi aplicam min-max.
* **Log1p + Normalizare L2:** aplicam aceeasi logaritmare, apoi normalizam fiecare casa.

PCA construieste **componente principale**, adica noi combinatii liniare ale atributelor procesate. `explained_variance_ratio_` indica proportia variantei explicate de fiecare componenta, iar suma cumulativa determina numarul minim de componente necesar pentru **98%, 95%, 90% si 80%**.

### Interpretarea Variantei Explicate

* **Curba cumulativa:** o crestere rapida indica faptul ca primele componente concentreaza o parte mare a variatiei.
* **Numarul de componente:** tabelul compara dimensionalitatea dupa encoding cu dimensionalitatea pastrata la fiecare prag.
* **Retinerea variantei:** 98% varianta nu inseamna 98% dintre coloane si nici o acuratete de 98%.
* **Efectul preprocesarii:** variantele pot necesita numere diferite de componente, deoarece scalarea si logaritmarea schimba distributia variantei intre atribute.

In [ ]:
experiments = {}
variance_rows = []
for use_log in [False, True]:
    for scaling in ['minmax', 'normalize']:
        name = scaling + (' + log1p' if use_log else '')
        prep = make_preprocessor()
        encoded = prep.fit_transform(before_encoding(X_raw, use_log))
        scaler = MinMaxScaler() if scaling == 'minmax' else Normalizer(norm='l2')
        scaled = scaler.fit_transform(encoded)
        pca = PCA(svd_solver='full')
        scores = pca.fit_transform(scaled)
        cumulative = np.cumsum(pca.explained_variance_ratio_)
        counts = {t: int(np.searchsorted(cumulative, t) + 1) for t in THRESHOLDS}
        experiments[name] = dict(prep=prep, scaler=scaler, pca=pca, scores=scores,
                                 counts=counts, use_log=use_log)
        for t, count in counts.items():
            variance_rows.append([name, t, encoded.shape[1], count, cumulative[count-1]])
        display(pd.DataFrame({'PC': np.arange(1, len(cumulative)+1),
                              'explained_ratio': pca.explained_variance_ratio_,
                              'cumulative': cumulative}).rename_axis(name))
        plt.plot(np.arange(1, len(cumulative)+1), cumulative, label=name)
for t in THRESHOLDS: plt.axhline(t, color='gray', linestyle=':', alpha=.5)
plt.xlabel('Number of principal components'); plt.ylabel('Cumulative explained variance')
plt.legend(); plt.tight_layout(); plt.show()
variance_table = pd.DataFrame(variance_rows, columns=['variant', 'threshold', 'encoded_features', 'components', 'retained_variance'])
display(variance_table)


### Interpretarea rezultatelor (varianta explicata)

* **Min-max si log1p + min-max (curbele albastra si verde):** curba creste lent. PC1 explica doar **12.2%**, iar primele 5 componente aproximativ **31%**. Sunt necesare **135 / 98 / 72 / 45** componente pentru **98% / 95% / 90% / 80%** din cele **287** coloane dupa encoding (log1p: 135 / 100 / 73 / 46).
* **De ce arata asa:** dupa min-max fiecare coloana are acelasi domeniu `[0, 1]`, deci contribuie comparabil la varianta. Majoritatea celor 287 de coloane sunt **one-hot**, slab corelate intre ele, iar o coloana binara echilibrata are varianta mare (pana la 0.25). Atributele continue cu valori extreme sunt comprimate spre 0 de min-max si au varianta mica. Varianta este astfel imprastiata in multe directii independente si nu exista cateva componente care sa rezume casele. Totusi, ultimele ~150 de componente adauga sub 2% din varianta, deci reducerea cu 50–85% este justificata.
* **Normalizare L2 (curbele portocalie si rosie):** curba urca aproape vertical: PC1 explica **74.5%**, iar **5** componente ajung la 95%. Motivul este ca `Normalizer` imparte fiecare rand la norma sa, fara sa egalizeze scalele coloanelor. Norma este dominata de cateva atribute cu valori mari (`Lot Area` ~10 000, cei patru ani ~2 000, suprafetele), iar coloanele one-hot si notele de calitate (0–10) devin practic zero. Datele traiesc deci intr-un spatiu cu foarte putine dimensiuni efective.
* **Logaritmarea + L2:** dupa `log1p`, `Lot Area` scade la ~9 si nu mai domina vectorul. Varianta se imparte intre PC1, PC2 si PC3 (**33% / 32% / 25%**), determinate de ani si suprafete. Pragurile de 95% si 90% cer acelasi numar de componente (4), deoarece PC4 sare direct de la 89.95% la 95.4%.
* **Concluzie:** numarul de componente spune mai mult despre **preprocesare** decat despre complexitatea caselor. Un numar mic de componente la L2 nu inseamna o reprezentare mai buna, ci faptul ca majoritatea atributelor au fost eliminate implicit.

## 4. Antrenarea K-Means si Alegerea Numarului de Clustere

Pentru fiecare combinatie dintre preprocesare si prag PCA, antrenam modele **K-means** cu mai multe valori ale lui `k`. Fiecare casa este atribuita unui centroid, iar algoritmul urmareste reducerea distantelor patratice din interiorul grupurilor.

* **Elbow Method:** reprezentam `inertia_` pentru `k = 1...10`. Cautam un cot dupa care adaugarea clusterelor aduce imbunatatiri mai mici.
* **Silhouette Score:** pentru `k = 2...10`, comparam apropierea caselor de propriul cluster cu apropierea de alte clustere.
* **Alegerea implicita:** selectam valoarea lui `k` cu cel mai mare scor silhouette dintre cele testate. Alegerea poate fi ajustata prin `K_OVERRIDES`, pe baza graficului elbow si a interpretarii grupurilor.
* **Initializare:** `n_init=10` incearca mai multe initializari ale centroizilor pentru fiecare model.

### Interpretarea Alegerii lui k

* **Inertia scade cand creste k:** aceasta scadere, singura, nu justifica alegerea unui numar mare de clustere.
* **Elbow si silhouette:** folosim impreuna cele doua criterii; un maxim silhouette nu garanteaza grupuri utile pentru interpretarea caselor.
* **Comparatia intre variante:** inertia depinde de scara si dimensionalitatea datelor, deci valorile brute nu sunt un clasament corect intre preprocesari diferite.

In [ ]:
K_OVERRIDES = {}  # Example: {('minmax', 0.98): 3}
results = {}; comparison_rows = []
for name, exp in experiments.items():
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    for col, t in enumerate(THRESHOLDS):
        z = exp['scores'][:, :exp['counts'][t]]
        models = {}; inertias = []; sil_scores = []
        for k in [1] + K_VALUES:
            model = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(z)
            models[k] = model; inertias.append(model.inertia_)
            if k > 1: sil_scores.append(silhouette_score(z, model.labels_))
        best_k = K_OVERRIDES.get((name, t), K_VALUES[int(np.argmax(sil_scores))])
        assert best_k in K_VALUES
        results[(name, t)] = dict(z=z, models=models, k=best_k, model=models[best_k])
        comparison_rows.append([name, t, z.shape[1], best_k, sil_scores[K_VALUES.index(best_k)]])
        axes[0,col].plot([1]+K_VALUES, inertias, 'o-')
        axes[0,col].set(title=f'{t:.0%} variance', xlabel='k', ylabel='Inertia')
        axes[1,col].plot(K_VALUES, sil_scores, 'o-')
        axes[1,col].axvline(best_k, color='red', linestyle='--')
        axes[1,col].set(xlabel='k', ylabel='Mean silhouette')
    fig.suptitle(name); plt.tight_layout(); plt.show()
comparison = pd.DataFrame(comparison_rows, columns=['variant', 'threshold', 'components', 'k', 'silhouette'])
display(comparison)


### Interpretarea rezultatelor (Elbow si Silhouette)

* **Min-max:** inertia scade lin, fara un cot clar. Exista doar o curbura slaba la k = 3–4. Silhouette are maximul la **k = 3** pentru 98%, 95% si 90% (≈ **0.094 – 0.102**) si scade brusc la k = 4. La 80%, k = 2 (0.1155) si k = 3 (0.1136) sunt practic la egalitate, iar regula maximului alege k = 2.
* **De ce valorile silhouette sunt atat de mici:** in spatiul min-max casele formeaza un **continuum** (de la case vechi la case noi) si nu grupuri compacte separate de spatii goale. In zeci de dimensiuni distantele intre puncte devin asemanatoare, deci diferenta dintre distanta intra-cluster si distanta catre clusterul vecin este mica. Valori de ~0.1 indica o structura slaba, dar nu inexistenta.
* **Normalizare L2:** cot foarte clar la **k = 2** si silhouette mare (**0.61 – 0.69**), urmat de o cadere la ~0.25–0.40. Datele L2 au un nor principal dens si un grup mic, departat, de case cu teren foarte mic (vezi sectiunea 5). Un astfel de grup izolat produce un silhouette mare la k = 2, chiar daca impartirea este dezechilibrata.
* **Normalizare L2 + log1p:** silhouette moderat (**0.31 – 0.36**), cu maxime la k = 4 (98%) si k = 8 (95%, 90%, 80%). Curba silhouette este aproape plata intre k = 4 si k = 8, deci datele nu au un numar „natural” de clustere: K-means doar imparte un nor continuu in felii. Alegerea k = 8 trebuie privita cu prudenta.
* **Inertia nu este comparabila intre variante:** valorile ~30 000 la min-max si ~20–140 la L2 provin din scari diferite ale datelor, nu din calitatea gruparii.

### Evaluarea Detaliata prin Diagrame Silhouette

Analizam distributia scorurilor individuale pentru **k = 2, 3, 4** si pentru valoarea selectata anterior, fara a repeta valorile identice.

* **Scor apropiat de 1:** casa este bine separata de celelalte grupuri.
* **Scor apropiat de 0:** casa se afla in apropierea granitei dintre clustere.
* **Scor negativ:** casa este, in medie, mai apropiata de un alt cluster decat de cel atribuit.
* **Grosimea benzilor:** arata numarul de case din fiecare grup si permite observarea clusterelor foarte mici sau dezechilibrate.
* **Linia rosie:** reprezinta media scorurilor silhouette pentru modelul respectiv.

Diagramele completeaza media folosita anterior: doua modele cu medii apropiate pot avea distributii diferite ale scorurilor si grupuri cu niveluri diferite de separare.

In [ ]:
# Full silhouette diagrams for several k values in every representation.
for (name, t), result in results.items():
    candidates = sorted(set([2, 3, 4, result['k']]))
    fig, axes = plt.subplots(1, len(candidates), figsize=(4*len(candidates), 4), squeeze=False)
    for ax, k in zip(axes[0], candidates):
        labels = result['models'][k].labels_
        values = silhouette_samples(result['z'], labels)
        bottom = 0
        for group in range(k):
            v = np.sort(values[labels == group]); top = bottom + len(v)
            ax.fill_betweenx(np.arange(bottom, top), 0, v, alpha=.7)
            ax.text(-.08, (bottom+top)/2, str(group))
            bottom = top + 10
        ax.axvline(values.mean(), color='red', linestyle='--')
        ax.set(title=f'k={k}, mean={values.mean():.3f}', xlabel='Silhouette', ylabel='Houses by cluster', xlim=(-1,1))
    fig.suptitle(f'{name}, retained variance {t:.0%}'); plt.tight_layout(); plt.show()


### Interpretarea rezultatelor (diagrame Silhouette)

* **Min-max:** toate benzile sunt inguste si concentrate intre **0 si ~0.25**, iar fiecare cluster are la baza cateva case cu scor negativ. Casele de la marginea clusterelor sunt aproape la fel de apropiate de grupul vecin. La k = 3 benzile au grosimi apropiate (~800 / ~1000 / ~1100 case), deci impartirea este echilibrata. La k = 4 media scade, iar mai multe case ajung la valori negative: al patrulea cluster taie un grup existent, fara o granita reala.
* **Normalizare L2, k = 2:** banda clusterului mare (~2580 case) are valori de **0.6 – 0.75**, deoarece casele sunt foarte apropiate de propriul centroid si departe de grupul mic. Clusterul mic (~345 case) are scoruri mai mici si mai variabile, pentru ca este imprastiat. Media mare este data in principal de clusterul mare, nu de o separare buna a ambelor grupuri.
* **Normalizare L2 + log1p:** la k = 4 si k = 8 aproape toate casele au scoruri pozitive (0 – 0.6), iar benzile au grosimi comparabile. Exceptie este un cluster mic (~108 case) la k = 8. Separarea este mai buna decat la min-max, dar se face pe putine dimensiuni (ani si suprafete).
* **Concluzie:** media silhouette ascunde diferente importante. La min-max separarea este slaba, dar uniforma intre grupuri. La L2 este puternica pentru un singur grup mare.

## 5. Vizualizarea Clusterelor si Compararea Rezultatelor

Reprezentam casele in planul format de **PC1 si PC2**, folosind culori diferite pentru clustere si simbolul **X** pentru centroizi. K-means foloseste toate componentele pastrate; primele doua sunt utilizate doar pentru vizualizare. Daca pragul pastreaza o singura componenta, a doua coordonata este zero.

### Interpretarea Graficelor si a Metricii ARI

* **Suprapunerea in doua dimensiuni:** grupurile care par suprapuse pot fi separate pe alte componente folosite de model.
* **Numerele clusterelor:** etichetele sunt arbitrare; clusterul 0 dintr-un model nu reprezinta automat acelasi tip de case in alt model.
* **Adjusted Rand Index (ARI):** comparam fiecare impartire cu varianta **min-max, 98% varianta**, fara a necesita aceleasi numere de cluster.
* **Valorile ARI:** 1 indica aceeasi impartire; valori in jurul lui 0 indica acord apropiat de cel asteptat intamplator; valorile negative indica un acord mai slab.
* **Efectul logaritmarii si scalarii:** logaritmarea comprima masuratorile foarte mari, min-max rescaleaza coloanele, iar normalizarea L2 modifica lungimea fiecarui rand. Aceste transformari pot schimba vecinii unei case si grupul atribuit.

ARI masoara asemanarea intre impartiri, nu stabileste singur care model este mai bun.

In [ ]:
for name, exp in experiments.items():
    fig, axes = plt.subplots(1, 4, figsize=(19, 4))
    for ax, t in zip(axes, THRESHOLDS):
        r = results[(name, t)]; z = r['z']; model = r['model']
        y = z[:,1] if z.shape[1] > 1 else np.zeros(len(z))
        centers = model.cluster_centers_
        cy = centers[:,1] if z.shape[1] > 1 else np.zeros(len(centers))
        ax.scatter(z[:,0], y, c=model.labels_, cmap='tab10', s=10, alpha=.5)
        ax.scatter(centers[:,0], cy, c='black', marker='X', s=140, label='Centers')
        ax.set(title=f'{t:.0%}: {z.shape[1]} PCs, k={r["k"]}', xlabel='PC1', ylabel='PC2')
        ax.legend()
    fig.suptitle(name); plt.tight_layout(); plt.show()
# ARI compares memberships without requiring matching cluster numbers.
reference = results[('minmax', .98)]['model'].labels_
comparison['ARI_vs_minmax_98'] = [adjusted_rand_score(reference, results[(row.variant,row.threshold)]['model'].labels_)
                                 for row in comparison.itertuples()]
display(comparison)


### Interpretarea rezultatelor (PC1–PC2 si ARI)

* **Min-max (98%, 95%, 90%):** in planul PC1–PC2 se disting trei regiuni: **stanga-jos** (PC1 negativ, PC2 negativ), **stanga-sus** (PC2 pozitiv) si **dreapta** (PC1 pozitiv). Granitele sunt reale, dar norii se ating, ceea ce explica silhouette-ul mic. Culorile se schimba intre praguri doar pentru ca etichetele sunt arbitrare: impartirea este aceeasi (**ARI = 0.998**). La 80% (k = 2), cele doua regiuni din stanga se unesc (**ARI = 0.577**).
* **De ce pragul conteaza putin:** K-means foloseste distante euclidiene, iar acestea sunt dominate de primele componente, care au varianta mare. Componentele eliminate intre 98% si 90% au varianta mica si schimba foarte putin distantele, deci si centroidul cel mai apropiat.
* **Normalizare L2:** un nor dens in dreapta (PC1 ≈ 0 – 0.4) si un grup imprastiat in stanga (PC1 < −0.2). PC1 este axa „teren mare vs. ani” (sectiunea 6): o casa cu teren foarte mic are vectorul normalizat dominat de ani, deci se muta spre stanga. ARI ≈ **0.02** fata de min-max: cele doua metode grupeaza dupa criterii complet diferite.
* **Normalizare L2 + log1p:** apar **dungi liniare**. Ele corespund caselor care au zero la un atribut de suprafata (fara etaj 2 sau fara subsol): dupa impartirea la norma, aceste case se aliniaza pe aceeasi directie. K-means taie norul in 4 sau 8 felii aproximativ paralele. ARI ≈ **0.12 – 0.14** indica o suprapunere mica cu gruparea min-max.
* **Log1p + min-max:** ARI = **0.981** la 98% si 95%, deci logaritmarea muta doar cateva case. La 90% si 80% se alege k = 2 (ARI 0.565), aceeasi unire a grupurilor din stanga ca la min-max 80%.

## 6. Influenta Atributelor asupra Componentelor Principale

Calculam **loadings** folosind formula ceruta: `pca.components_.T * np.sqrt(pca.explained_variance_)`. Reprezentam contributiile atributelor importante pentru primele cinci componente si afisam separat atributele dominante pentru primele trei.

### Interpretarea Loadings

* **Magnitudinea contributiei:** o valoare absoluta mare indica o contributie puternica in reprezentarea calculata.
* **Semnul contributiei:** atributele cu semne opuse contribuie in directii opuse la coordonata componentei respective.
* **Heatmap:** selectam pana la 18 atribute cu contributii absolute mari, pentru a pastra graficul lizibil.
* **Semnul componentelor PCA:** o componenta poate avea toate semnele inversate fara sa se schimbe informatia pe care o reprezinta.
* **Legatura cu datele originale:** loadings descriu spatiul transformat; nu trebuie interpretate automat drept corelatii cu masuratorile originale nescalate.

Atributele dominante ne ajuta sa explicam ce diferente dintre case sunt surprinse de PC1, PC2 si PC3.

In [ ]:
for name, exp in experiments.items():
    pca = exp['pca']; names = exp['prep'].get_feature_names_out()
    count = min(5, len(pca.components_))
    loadings = pd.DataFrame(pca.components_.T * np.sqrt(pca.explained_variance_), index=names)
    top = loadings.iloc[:,:count].abs().max(axis=1).nlargest(18).index
    plt.figure(figsize=(9,8))
    sns.heatmap(loadings.loc[top].iloc[:,:count], cmap='coolwarm', center=0,
                xticklabels=[f'PC{i+1}' for i in range(count)])
    plt.title(f'Leading PCA loadings — {name}'); plt.tight_layout(); plt.show()
    for pc in range(min(3, count)):
        print(name, f'PC{pc+1}:', ', '.join(loadings.iloc[:,pc].abs().nlargest(5).index))


### Interpretarea rezultatelor (loadings)

* **Min-max, PC1 — „modern vs. vechi”:** contributii pozitive mari pentru `Foundation_PConc`, `Exterior VinylSd`, `Garage Type_Attchd`, `BsmtFin Type 1_GLQ`, `Year Remod/Add` si negative pentru `Garage Finish_Unf`, `Garage Type_Detchd`, `Foundation_CBlock`. Toate descriu epoca constructiei: casele noi au fundatie din beton turnat, siding vinil si garaj atasat, iar cele vechi au garaj detasat si nefinisat.
* **Min-max, PC2 si PC3 — „un nivel vs. doua niveluri”:** `House Style_1Story`, `MS SubClass_20` si `Foundation_CBlock` sunt opuse fata de `House Style_2Story`. PC2 separa casele tip ranch cu un nivel (tipice anilor '50–'70) de cele cu etaj.
* **De ce domina coloanele one-hot:** dupa min-max, o coloana binara cu frecventa apropiata de 50% are cea mai mare varianta posibila, iar atributele continue sunt comprimate de valorile extreme. PCA urmareste varianta, deci directiile principale sunt formate din categorii frecvente si corelate intre ele (aceeasi epoca determina simultan fundatia, sidingul si garajul).
* **Log1p + min-max:** aceleasi atribute dominante. Logaritmarea a afectat doar 11 coloane numerice, iar acestea au oricum contributie mica.
* **Normalizare L2:** PC1 opune `Lot Area` celor patru atribute de tip an (`Yr Sold`, `Year Built`, `Year Remod/Add`, `Garage Yr Blt`). PC2–PC3 sunt formate din suprafete (`Total Bsmt SF`, `1st/2nd Flr SF`, `Bsmt Unf SF`). Calitatea, cartierul si stilul nu apar deloc: au valori mici fata de norma randului.
* **Normalizare L2 + log1p:** fara `Lot Area`, toate componentele principale descriu **suprafete si proportii** (subsol vs. etaj 2, subsol finisat vs. nefinisat), plus ani. Valorile loadings sunt mici (sub 0.08), deoarece toate datele L2 au varianta totala mica.

## 7. Caracterizarea Clusterelor pe Baza Proprietatilor Caselor

Atasam etichetele de cluster la datele originale si calculam statistici pentru **suprafata locuibila, suprafata terenului, calitate, anul constructiei si pret**. Pentru fiecare grup afisam numarul de case si cartierul cel mai frecvent.

### Interpretarea Profilurilor si a Graficelor

* **Mediana:** descrie valoarea centrala a unui atribut si este mai putin influentata de valori extreme decat media.
* **Profilul unui cluster:** combinam caracteristicile, de exemplu suprafata, vechimea si calitatea, pentru a descrie tipurile de case din grup. Descrierea concreta trebuie sustinuta de valorile afisate.
* **Boxploturi:** comparam distributiile si variatia din interiorul clusterelor, nu doar valorile centrale.
* **Distributia cartierelor:** heatmapul arata proportia fiecarui cartier in interiorul unui cluster.
* **Pretul de vanzare:** este folosit doar pentru interpretare dupa grupare, fara sa participe la antrenarea PCA sau K-means.

Graficele detaliate folosesc implicit **min-max cu 98% varianta**. `PROFILE_KEY` permite inspectarea unei alte variante, iar tabelele rezuma toate modelele selectate.

In [ ]:
PROFILE_COLS = ['Gr Liv Area', 'Lot Area', 'Overall Qual', 'Year Built', 'SalePrice']
for key, r in results.items():
    profile = df.assign(cluster=r['model'].labels_)
    print(key)
    display(profile.groupby('cluster')[PROFILE_COLS].median().round(1).join(profile.groupby('cluster').size().rename('count')))
    for group, houses in profile.groupby('cluster'):
        med = houses[PROFILE_COLS].median()
        neighborhood = houses['Neighborhood'].mode().iloc[0]
        print(f'Cluster {group}: {len(houses)} houses; median living area {med["Gr Liv Area"]:.0f} sq ft, '
              f'quality {med["Overall Qual"]:.1f}, construction year {med["Year Built"]:.0f}, '
              f'price ${med["SalePrice"]:,.0f}; most common neighborhood {neighborhood}.')
PROFILE_KEY = ('minmax', .98)
profile = df.assign(cluster=results[PROFILE_KEY]['model'].labels_)
fig, axes = plt.subplots(1, 4, figsize=(18,4))
for ax, feature in zip(axes, ['Gr Liv Area', 'Overall Qual', 'Year Built', 'SalePrice']):
    sns.boxplot(data=profile, x='cluster', y=feature, ax=ax)
plt.tight_layout(); plt.show()
plt.figure(figsize=(12,5))
sns.heatmap(pd.crosstab(profile['cluster'], profile['Neighborhood'], normalize='index'), cmap='Blues')
plt.title('Neighborhood proportions within each cluster'); plt.tight_layout(); plt.show()


### Interpretarea rezultatelor (profilurile clusterelor, min-max 98%, k = 3)

| Cluster | Case | An median | Suprafata | Teren | Calitate | Pret median | Cartiere dominante | Profil |
|---|---|---|---|---|---|---|---|---|
| **0** | 1034 | 1966 | 1222 sqft | 9894 | 5 | \$146 500 | NAmes, Sawyer, NWAmes | case suburbane cu un nivel din anii '50–'70 |
| **1** | 803 | 1932 | 1322 sqft | 7500 | 5 | \$123 500 | OldTown, BrkSide, IDOTRR, Edwards | case vechi din zona centrala, loturi mici |
| **2** | 1090 | 2003 | 1665 sqft | 9982 | 7 | \$222 000 | CollgCr, NridgHt, Somerst, Gilbert | case moderne, calitate ridicata |

* **Boxploturi:** `Year Built` separa cel mai clar grupurile: clusterul 2 are o cutie foarte ingusta (~1998–2006), clusterul 0 ~1958–1976, iar clusterul 1 ~1920–1950. Suprafata locuibila se suprapune puternic intre clustere, deci **marimea nu este criteriul principal**. Calitatea separa doar clusterul 2 (7–8) de celelalte doua (5–6).
* **Pretul** nu a participat la antrenare, dar este ordonat clar: \$123k < \$147k < \$222k. Clusterul 2 are cea mai mare dispersie de pret (de la ~\$85k la peste \$700k), deoarece contine atat case noi obisnuite, cat si case de lux. K-means nu a creat un grup separat pentru lux, pentru ca acestea difera mai ales prin suprafata si calitate, nu prin atributele categorice care domina distantele.
* **Outlierii din clusterul 1 (case construite dupa 1990):** sunt case noi cu caracteristici „vechi” (garaj detasat, fundatie diferita de beton turnat, fara siding vinil). Modelul grupeaza dupa **stilul constructiei**, nu direct dupa an.
* **Heatmap-ul cartierelor:** fiecare cluster este dominat de cartiere diferite, aproape fara suprapuneri: NAmes (~37%) in clusterul 0, OldTown (~27%) in clusterul 1, CollgCr (~20%) in clusterul 2. Cartierele din Ames au fost construite in epoci diferite, deci epoca, stilul si cartierul sunt puternic corelate.
* **Celelalte variante:** min-max la 95% si 90% si log1p + min-max la 98% si 95% reproduc aceleasi trei profiluri. La 80% grupurile 0 si 1 se unesc (1958, \$136 500). L2 separa 344 de case cu teren foarte mic (**3632 sqft**, cartier dominant Somerst, tip townhouse), cu calitate si pret aproape egale cu restul. L2 + log1p imparte casele dupa **marime si vechime** (de exemplu, grupul cu 2126 sqft si \$238 000 din NoRidge vs. grupul cu 995 sqft si \$114 500 din OldTown).

## 8. Inferenta pentru Case Noi

Construim **trei case sintetice** pornind de la exemple reale de dimensiuni diferite. Modificam suprafetele, terenul si calitatea si recalculam suprafata locuibila din suprafetele etajelor. Celelalte atribute sunt preluate din exemplele de baza, pentru a pastra descrieri complete.

1. **Preprocesare identica:** aplicam curatarea si eventuala logaritmare, apoi encoderul si scalerul deja antrenate.
2. **Transformare PCA:** folosim `transform` si retinem exact numarul de componente folosit la antrenarea modelului respectiv.
3. **Atribuirea clusterului:** `predict` selecteaza centroidul cel mai apropiat in spatiul PCA.
4. **Verificarea intuitiva:** comparam proprietatile casei inventate cu medianele clusterului atribuit in varianta de referinta.

### Interpretarea Predictiilor

* **De ce aplicam PCA inainte de predictie:** centroizii sunt exprimati in coordonate PCA, deci masuratorile brute ale unei case nu pot fi comparate direct cu acestia.
* **De ce nu folosim din nou fit:** reantrenarea transformarilor ar schimba sistemul de coordonate pe care modelul il asteapta.
* **Comparatia intre modele:** numerele clusterelor nu au aceeasi semnificatie intre variante; interpretam apartenenta folosind profilurile grupurilor.
* **Limita exemplelor:** acestea sunt variante sintetice ale unor observatii existente, nu un set independent pentru evaluarea generalizarii.

In [ ]:
# Start with low-, middle-, and high-size templates, then invent new measurements.
ordered = X_raw.sort_values('Gr Liv Area')
new_houses = ordered.iloc[[int((len(ordered)-1)*q) for q in [.2,.5,.8]]].copy()
new_houses.index = ['Modest house', 'Medium house', 'Large house']
for label, factor, quality in zip(new_houses.index, [.92, 1.03, 1.10], [4,6,8]):
    for col in ['1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF']:
        new_houses.loc[label,col] = round(new_houses.loc[label,col] * factor)
    new_houses.loc[label,'Gr Liv Area'] = new_houses.loc[label,['1st Flr SF','2nd Flr SF','Low Qual Fin SF']].sum()
    new_houses.loc[label,'Overall Qual'] = quality
    new_houses.loc[label,'Lot Area'] = round(new_houses.loc[label,'Lot Area'] * factor)
display(new_houses[['Gr Liv Area','Lot Area','Overall Qual','Year Built','Neighborhood']])
assignments = pd.DataFrame(index=new_houses.index)
for (name,t), r in results.items():
    exp = experiments[name]
    encoded = exp['prep'].transform(before_encoding(new_houses, exp['use_log']))
    scaled = exp['scaler'].transform(encoded)
    z_new = exp['pca'].transform(scaled)[:, :exp['counts'][t]]
    assignments[f'{name} / {t:.0%}'] = r['model'].predict(z_new)
display(assignments)
# Inspect assigned reference-cluster medians next to each synthetic house.
labels = assignments[f'{PROFILE_KEY[0]} / {PROFILE_KEY[1]:.0%}']
medians = profile.groupby('cluster')[PROFILE_COLS].median()
for house, group in labels.items():
    print(f'{house} -> cluster {group}')
    display(pd.DataFrame({'invented_house': new_houses.loc[house, PROFILE_COLS[:-1]],
                          'cluster_median': medians.loc[group, PROFILE_COLS[:-1]]}))
# Place the synthetic houses on the PC1-PC2 plane of the reference model.
ref = results[PROFILE_KEY]; ref_exp = experiments[PROFILE_KEY[0]]
new_Z = ref_exp['pca'].transform(ref_exp['scaler'].transform(
    ref_exp['prep'].transform(before_encoding(new_houses, ref_exp['use_log']))))[:, :ref['z'].shape[1]]
if ref['z'].shape[1] >= 2:
    Z = ref['z']; centers = ref['model'].cluster_centers_
    plt.figure(figsize=(8, 6))
    plt.scatter(Z[:, 0], Z[:, 1], c=ref['model'].labels_, cmap='tab10', s=10, alpha=.25)
    plt.scatter(centers[:, 0], centers[:, 1], c='black', marker='X', s=150, label='Centroizi')
    plt.scatter(new_Z[:, 0], new_Z[:, 1], c='red', marker='*', s=300, edgecolor='black', label='Case noi')
    for (x, y), house in zip(new_Z[:, :2], new_houses.index):
        plt.annotate(house, (x, y), xytext=(6, 6), textcoords='offset points')
    plt.xlabel('PC1'); plt.ylabel('PC2')
    plt.title(f'Casele inventate in spatiul PCA ({PROFILE_KEY[0]}, {PROFILE_KEY[1]:.0%})')
    plt.legend(); plt.tight_layout(); plt.show()


### Interpretarea rezultatelor (case noi)

| Casa | Atribute principale | Cluster (min-max 98%) | Profilul clusterului | Corect intuitiv? |
|---|---|---|---|---|
| Modest house | 979 sqft, Q4, 1952, NAmes | **0** | 1966, NAmes, 1222 sqft, Q5 | **Da, la granita.** Cartierul este cel dominant in cluster, iar anul 1952 se afla intre 1932 si 1966. In grafic este in stanga clusterului 0 (PC1 ≈ −1.6), aproape de clusterul caselor vechi. |
| Medium house | 1485 sqft, Q6, 1970, NWAmes | **0** | 1966, 1222 sqft, Q5 | **Da.** Anul si cartierul corespund clusterului 0. Suprafata si calitatea sunt peste mediana, de aceea casa apare in partea dreapta-sus, spre granita cu clusterul 2. |
| Large house | 2021 sqft, Q8, 1997, SawyerW | **2** | 2003, 1665 sqft, Q7 | **Da.** Este o casa moderna dintr-un cartier al clusterului 2 si se afla in interiorul norului acestuia, aproape de centroid. |

* **Consistenta intre variante:** toate modelele min-max si log1p + min-max pun casa modesta si cea medie in acelasi grup (anii '60 sau grupul unit „vechi”), iar casa mare in grupul modern. L2 pune toate cele trei case in clusterul mare, deoarece niciuna nu are teren foarte mic. L2 + log1p le separa dupa marime: casa modesta ajunge in grupul caselor mici si vechi, iar casa mare intr-un grup de case mari. Casa medie (1970) este atribuita uneori unui grup cu an median ~2000, ceea ce arata ca aceasta varianta ignora vechimea cand proportiile suprafetelor sunt asemanatoare.
* **De ce rezultatele arata asa:** casele sintetice pastreaza atributele categorice ale exemplelor reale (fundatie, garaj, siding, cartier). Modificarea suprafetei si a calitatii schimba putin pozitia in spatiul min-max, unde distantele sunt dominate de categorii. Din acest motiv casele raman in clusterul epocii lor si doar se apropie de granite.

## 9. Sinteza Analizei

Analiza leaga **preprocesarea**, **reducerea dimensionalitatii** si **gruparea caselor**. Concluziile de mai jos se bazeaza pe rularea cu `SEED = 42`.

* **Dimensionalitate:** din **287** atribute dupa encoding, min-max are nevoie de **135 / 98 / 72 / 45** componente pentru **98% / 95% / 90% / 80%**, iar PC1 explica doar 12.2%. Varianta este imprastiata intre multe coloane one-hot slab corelate. L2 ajunge la aceleasi praguri cu doar **8 / 5 / 4 / 2** componente (L2 + log1p: 5 / 4 / 4 / 3), dar numai pentru ca normalizarea pe randuri lasa sa conteze doar cateva atribute cu valori mari (`Lot Area`, anii, suprafetele).
* **Pragul PCA:** la min-max, pragurile de 98%, 95% si 90% produc aceeasi impartire (**ARI ≥ 0.998**). Componentele eliminate au varianta mica si nu schimba distantele. Doar la 80% (45 de componente) doua grupuri se unesc (k = 2, ARI 0.577). Pragul de **90% (72 de componente)** pastreaza rezultatul si reduce dimensiunea de aproximativ 4 ori.
* **Alegerea clusterelor:** la min-max, curba elbow este neteda, cu un cot slab la k = 3–4. Silhouette are maximul la **k = 3** (0.094–0.102) si scade brusc la k = 4. Alegem **k = 3**, sustinut de ambele criterii, de dimensiunile echilibrate ale grupurilor (~800 / ~1035 / ~1090) si de profilurile clare. Valorile silhouette mici arata ca grupurile sunt regiuni ale unui continuum, nu clustere bine separate.
* **Cele trei tipuri de case (min-max, 98%, k = 3):**
  * **case vechi din zona centrala** (1932, OldTown, teren 7500, calitate 5, \$123 500);
  * **case suburbane cu un nivel din anii '60** (1966, NAmes, 1222 sqft, calitate 5, \$146 500);
  * **case moderne** (2003, CollgCr, 1665 sqft, calitate 7, \$222 000).

  Pretul nu a participat la antrenare, dar este ordonat clar intre grupuri. Clusteringul surprinde deci o structura legata de valoarea casei.
* **Interpretabilitate:** loadings arata ca PC1 este axa **„modern vs. vechi”** (`Foundation_PConc`, siding vinil, garaj atasat vs. garaj detasat si nefinisat), iar PC2 este axa **„un nivel vs. doua niveluri”**. Clusterele sunt definite mai ales de **epoca si stilul constructiei**. Suprafata locuibila se suprapune mult intre grupuri, iar casele de lux nu formeaza un grup separat.
* **Preprocesare:**
  * **L2** obtine cel mai mare silhouette (**0.61–0.69**), dar imparte casele in 2583 vs. 344 si izoleaza doar casele cu teren foarte mic (~3630 sqft, Somerst). Calitatea si pretul sunt aproape egale intre grupuri, iar ARI fata de min-max este ≈ 0.02.
  * **L2 + log1p** grupeaza dupa marime si proportiile suprafetelor (silhouette 0.31–0.36, k = 4 sau 8). Curba silhouette este insa aproape plata, deci numarul de grupuri nu este bine determinat.
  * **Log1p + min-max** schimba foarte putin rezultatul (ARI 0.981 la 98% si 95%), deoarece au fost logaritmate doar 11 coloane numerice cu contributie mica.
  * **Min-max** ramane reprezentarea cea mai potrivita pentru interpretare, deoarece pastreaza contributia tuturor atributelor.
* **Case noi:** casa modesta (1952, NAmes) si cea medie (1970, NWAmes) sunt atribuite grupului din anii '60, iar casa mare (1997, SawyerW, calitate 8) grupului modern. Atribuirile sunt compatibile cu anul, cartierul si calitatea. Graficul arata insa ca primele doua case sunt aproape de granitele cu grupurile vecine, ceea ce confirma separarea slaba dintre clustere.

Un scor silhouette mare nu este suficient pentru alegerea finala. L2 are cel mai mare scor, dar cea mai putin utila impartire. Min-max are scoruri mici, dar produce grupuri echilibrate, interpretabile si legate de pret.